# Qualidade de `pagamentos.csv`

Diagnóstico do log de cobranças pelos mesmos cinco critérios usados em `eventos_assinatura.csv`:

| Critério | Pergunta |
|---|---|
| **Completude** | Está tudo lá? Campos preenchidos e cobranças que deveriam existir |
| **Validade** | Os valores estão no formato e no domínio esperados? |
| **Consistência** | Os dados concordam entre si, dentro do arquivo e com os eventos de assinatura? |
| **Unicidade** | Cada cobrança aparece uma vez só? |
| **Temporalidade** | As datas estão no período certo, sem buracos, no ritmo esperado? |

**Só diagnóstico:** nada é corrigido aqui. Cada achado vai para uma tabela-resumo em `saidas/`, com status
(`ok`, `esperado` ou `atenção`) e tratamento sugerido. Nada em `dados/brutos/` é alterado (hash conferido no início e no fim).

**Decisões já registradas que valem aqui** (`decisoes.md`): datas convertidas para o horário de Brasília, cada formato com
seu fuso; casamento com os eventos por `user_id` e data, com tolerância de 1 dia.

In [1]:
import hashlib
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
SAIDAS = ROOT / "saidas"
INICIO_LOG = pd.Timestamp("2024-07-01")
CORTE = pd.Timestamp("2026-09-30")
FIM_CORTE = CORTE + pd.Timedelta(days=1)  # 01/10/2026 00:00, horário de Brasília
REAJUSTE = pd.Timestamp("2026-03-01")     # mensal de R$ 34,90 para R$ 39,90
TZ_BR = "America/Sao_Paulo"

ARQUIVOS = ["pagamentos.csv", "eventos_assinatura.csv", "usuarios.csv"]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

achados = []

def achado(criterio, checagem, resultado, linhas, tratamento, status=None):
    """status: ok (nada encontrado), esperado (encontrado, mas é comportamento do negócio) ou atenção.
    Sem status explícito: ok quando nenhuma linha é afetada, atenção caso contrário."""
    if status is None:
        status = "ok" if linhas == 0 else "atenção"
    achados.append({"criterio": criterio, "checagem": checagem, "status": status,
                    "resultado": resultado, "linhas_afetadas": int(linhas), "tratamento_sugerido": tratamento})

def para_brasilia(texto):
    """Datas terminadas em Z estão em UTC; as demais, em horário de Brasília. Devolve horário de Brasília sem fuso."""
    eh_utc = texto.str.endswith("Z")
    utc = pd.Series(pd.NaT, index=texto.index, dtype="datetime64[ns, UTC]")
    utc[eh_utc] = pd.to_datetime(texto[eh_utc], format="%Y-%m-%dT%H:%M:%SZ", errors="coerce", utc=True)
    utc[~eh_utc] = (pd.to_datetime(texto[~eh_utc], format="%Y-%m-%d %H:%M:%S", errors="coerce")
                    .dt.tz_localize(TZ_BR).dt.tz_convert("UTC"))
    return utc.dt.tz_convert(TZ_BR).dt.tz_localize(None)

def casa_por_data(base, outro, tolerancia_dias=1):
    """Para cada linha de base (user_id, data_br), diz se outro tem linha do mesmo user_id a até N dias."""
    m = base[["user_id", "data_br"]].reset_index().merge(
        outro[["user_id", "data_br"]].rename(columns={"data_br": "data_outro"}), on="user_id", how="left")
    m["ok"] = (m["data_outro"] - m["data_br"]).dt.days.abs() <= tolerancia_dias
    return m.groupby("index")["ok"].any().reindex(base.index, fill_value=False)

## 0. Leitura

Tudo como texto e sem converter vazio em nulo, para ver o arquivo exatamente como chegou.

In [2]:
pagamentos = pd.read_csv(BRUTOS / "pagamentos.csv", dtype=str, keep_default_na=False)
eventos = pd.read_csv(BRUTOS / "eventos_assinatura.csv", dtype=str, keep_default_na=False)
usuarios = pd.read_csv(BRUTOS / "usuarios.csv", dtype=str, usecols=["user_id", "email"])

assert len(pagamentos) == 165_548  # linhas segundo o dicionário
N = len(pagamentos)
print(f"pagamentos: {N:,} linhas x {pagamentos.shape[1]} colunas -> {list(pagamentos.columns)}")
pagamentos.sample(5, random_state=0)

pagamentos: 165,548 linhas x 7 colunas -> ['pagamento_id', 'user_id', 'data_hora', 'valor', 'metodo', 'status', 'tentativa']


,pagamento_id,user_id,data_hora,valor,metodo,status,tentativa
96461,p_0096461,u_a21953b7,2025-12-12T12:17:52Z,299.90,app_store,aprovado,1
90786,p_0090786,u_9c31f3a3,2025-11-17T16:43:19Z,299.90,app_store,aprovado,1
162137,p_0162137,u_bfe06cbb,2026-09-23 07:25:17,299.90,cartao,aprovado,1
40028,p_0040028,u_e3fd7316,2025-03-18 14:08:08,"R$ 34,90",cartao,aprovado,1
128311,p_0128311,u_4ca96ece,2026-04-21 07:23:51,39.90,pix_automatico,aprovado,1


### Colunas derivadas para a análise

- `dh_br`, `data_br`, `mes_br`: data em horário de Brasília;
- `valor_num`: o valor como número. O arquivo traz dois formatos, `34.90` e `R$ 34,90`.

O arquivo original não muda.

In [3]:
pagamentos["formato_data"] = pagamentos["data_hora"].str.endswith("Z").map({True: "UTC (Z)", False: "Brasília"})
pagamentos["dh_br"] = para_brasilia(pagamentos["data_hora"])
pagamentos["data_br"] = pagamentos["dh_br"].dt.normalize()
pagamentos["mes_br"] = pagamentos["dh_br"].dt.to_period("M")

pagamentos["formato_valor"] = pagamentos["valor"].str.startswith("R$").map({True: "R$ 0,00", False: "0.00"})
texto_valor = pagamentos["valor"].str.replace("R$", "", regex=False).str.strip()
eh_br = pagamentos["formato_valor"] == "R$ 0,00"
texto_valor[eh_br] = texto_valor[eh_br].str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
pagamentos["valor_num"] = pd.to_numeric(texto_valor, errors="coerce")

eventos["dh_br"] = para_brasilia(eventos["data_hora"])
eventos["data_br"] = eventos["dh_br"].dt.normalize()

pagamentos = pagamentos.sort_values(["user_id", "dh_br", "pagamento_id"], kind="stable")
pagamentos.drop_duplicates("formato_valor")[["valor", "formato_valor", "valor_num", "data_hora", "dh_br"]]

,valor,formato_valor,valor_num,data_hora,dh_br
28017,34.90,0.00,34.9,2025-01-14T19:34:00Z,2025-01-14 16:34:00
39563,"R$ 34,90","R$ 0,00",34.9,2025-03-16 06:19:00,2025-03-16 06:19:00


## 1. Completude

### 1.1 Campos preenchidos

In [4]:
VAZIOS_DISFARCADOS = {"nan", "null", "none", "na", "n/a", "-"}
colunas = ["pagamento_id", "user_id", "data_hora", "valor", "metodo", "status", "tentativa"]
vazios = pd.DataFrame({
    "vazio": [(pagamentos[c].str.strip() == "").sum() for c in colunas],
    "vazio_disfarcado": [pagamentos[c].str.strip().str.lower().isin(VAZIOS_DISFARCADOS).sum() for c in colunas],
}, index=colunas)
display(vazios)
achado("Completude", "campos vazios", f"{int(vazios.values.sum()):,} células vazias", vazios.values.sum(),
       "Nenhum")

,vazio,vazio_disfarcado
pagamento_id,0,0
user_id,0,0
data_hora,0,0
valor,0,0
metodo,0,0
status,0,0
tentativa,0,0


### 1.2 Cobranças que deveriam existir

Cada início e cada renovação em `eventos_assinatura.csv` é uma cobrança: deve haver um pagamento aprovado do mesmo usuário,
no mesmo dia (tolerância de 1 dia). O mesmo vale ao contrário, e para chargeback e estorno.

In [5]:
aprovados = pagamentos[pagamentos["status"] == "aprovado"]
ev_cobranca = eventos[eventos["evento"].isin(["assinatura_iniciada", "renovacao"])]

ev_sem_pag = ~casa_por_data(ev_cobranca, aprovados)
pag_sem_ev = ~casa_por_data(aprovados, ev_cobranca)

pares = [("chargeback", "chargeback"), ("estornado", "reembolso")]
linhas = []
for status, evento in pares:
    pag = pagamentos[pagamentos["status"] == status]
    tem = casa_por_data(pag, eventos[eventos["evento"] == evento])
    linhas.append({"pagamento": status, "evento": evento, "pagamentos": len(pag), "com evento": int(tem.sum()),
                   "sem evento": int((~tem).sum()), "eventos no log": int((eventos["evento"] == evento).sum())})

cobertura = pd.DataFrame([
    {"checagem": "início ou renovação sem pagamento aprovado", "esperado": len(ev_cobranca), "sem par": int(ev_sem_pag.sum())},
    {"checagem": "pagamento aprovado sem início ou renovação", "esperado": len(aprovados), "sem par": int(pag_sem_ev.sum())},
])
display(cobertura)
display(pd.DataFrame(linhas))

achado("Completude", "início ou renovação sem pagamento aprovado", f"{ev_sem_pag.sum():,} de {len(ev_cobranca):,} eventos",
       ev_sem_pag.sum(), "Nenhum")
achado("Completude", "pagamento aprovado sem início ou renovação", f"{pag_sem_ev.sum():,} de {len(aprovados):,} pagamentos",
       pag_sem_ev.sum(), "Nenhum")
sem_ev_estorno = sum(l["sem evento"] for l in linhas)
achado("Completude", "chargeback ou estorno sem evento correspondente",
       "; ".join(f"{l['pagamento']}: {l['com evento']:,} de {l['pagamentos']:,}" for l in linhas), sem_ev_estorno, "Nenhum")

,checagem,esperado,sem par
0,início ou renovação sem pagamento aprovado,159989,0
1,pagamento aprovado sem início ou renovação,161930,0


,pagamento,evento,pagamentos,com evento,sem evento,eventos no log
0,chargeback,chargeback,296,296,0,296
1,estornado,reembolso,72,72,0,72


Atenção: os números acima batem por **dia**, mas a quantidade não. Há mais pagamentos aprovados do que inícios e renovações.
A seção 4 (Unicidade) mostra de onde vem a diferença.

In [6]:
print(f"pagamentos aprovados: {len(aprovados):,} | inícios + renovações: {len(ev_cobranca):,} | "
      f"diferença: {len(aprovados) - len(ev_cobranca):,}")

pagamentos aprovados: 161,930 | inícios + renovações: 159,989 | diferença: 1,941


### 1.3 Valores com cupom

O dicionário diz que o arquivo inclui os valores com cupom. O enunciado cita dois cupons:
- **influenciador:** anual a R$ 239,90 no primeiro ano;
- **campanha de 10/08/2026:** mensal com 30% de desconto por 3 meses (R$ 27,93) ou anual com 1 mês grátis
  (receita abatida no fim do ciclo, que não aparece como valor de cobrança).

In [7]:
VALOR_CUPOM_CAMPANHA = round(39.90 * 0.7, 2)
cupom_influenciador = int((pagamentos["valor_num"] == 239.90).sum())
cupom_campanha = int((pagamentos["valor_num"].round(2) == VALOR_CUPOM_CAMPANHA).sum())
print(f"cobranças de R$ 239,90 (influenciador): {cupom_influenciador:,}")
print(f"cobranças de R$ {VALOR_CUPOM_CAMPANHA:.2f} (30% da campanha): {cupom_campanha:,}")
print("valores distintos no arquivo:", sorted(pagamentos["valor_num"].unique()))
achado("Completude", "cobrança com o desconto de 30% da campanha de agosto (R$ 27,93)",
       f"{cupom_campanha:,} cobranças; nenhum valor diferente dos preços de tabela e do cupom de influenciador",
       cupom_campanha, status="atenção" if cupom_campanha == 0 else "ok", tratamento=
       "Não dá para medir o custo do cupom da campanha por este arquivo. Conferir com campanha_ago26.csv "
       "quem recebeu a oferta; pode ser que ninguém tenha usado ou que o desconto não tenha sido registrado")

cobranças de R$ 239,90 (influenciador): 1,710
cobranças de R$ 27.93 (30% da campanha): 0
valores distintos no arquivo: [np.float64(-299.9), np.float64(-39.9), np.float64(34.9), np.float64(39.9), np.float64(239.9), np.float64(299.9)]


In [8]:
eh_teste = usuarios.set_index("user_id")["email"].str.strip().str.lower().str.endswith("@vitaliza.com.br")
pag_teste = int(pagamentos["user_id"].map(eh_teste).fillna(False).sum())
print(f"pagamentos de contas de teste (@vitaliza.com.br): {pag_teste:,}")
achado("Completude", "pagamentos de contas de teste", f"{pag_teste:,}", pag_teste, "Se houver: excluir")

pagamentos de contas de teste (@vitaliza.com.br): 0


## 2. Validade

In [9]:
DOMINIO = {
    "metodo": {"app_store", "google_play", "cartao", "pix_automatico"},
    "status": {"aprovado", "recusado", "estornado", "chargeback"},
}
for col, dominio in DOMINIO.items():
    fora = ~pagamentos[col].isin(dominio)
    print(f"{col}: {pagamentos[col].value_counts().to_dict()} | fora do dicionário: {fora.sum():,}")
    achado("Validade", f"{col} fora da lista do dicionário", f"{fora.sum():,} linhas", fora.sum(), "Nenhum")

tentativa_num = pd.to_numeric(pagamentos["tentativa"], errors="coerce")
tentativa_invalida = (tentativa_num.isna() | (tentativa_num < 1) | (tentativa_num % 1 != 0)).sum()
print(f"tentativa: {pagamentos['tentativa'].value_counts().to_dict()} | não é inteiro positivo: {tentativa_invalida:,}")
achado("Validade", "tentativa que não é inteiro positivo", f"{tentativa_invalida:,} linhas", tentativa_invalida, "Nenhum")

for col in ["metodo", "status", "user_id", "pagamento_id"]:
    sujo = (pagamentos[col] != pagamentos[col].str.strip().str.lower()).sum()
    print(f"{col}: {sujo:,} valores com maiúscula ou espaço nas pontas")

metodo: {'app_store': 57020, 'google_play': 55106, 'cartao': 34676, 'pix_automatico': 18746} | fora do dicionário: 0
status: {'aprovado': 161930, 'recusado': 3250, 'chargeback': 296, 'estornado': 72} | fora do dicionário: 0


tentativa: {'1': 160682, '2': 4866} | não é inteiro positivo: 0


metodo: 0 valores com maiúscula ou espaço nas pontas


status: 0 valores com maiúscula ou espaço nas pontas
user_id: 0 valores com maiúscula ou espaço nas pontas


pagamento_id: 0 valores com maiúscula ou espaço nas pontas


In [10]:
# valor: converte nos dois formatos? Está entre os preços conhecidos?
PRECOS = {34.90, 39.90, 299.90, 239.90}
nao_converte = int(pagamentos["valor_num"].isna().sum())
fora_preco = ~pagamentos["valor_num"].abs().round(2).isin(PRECOS)
print("formatos de valor:", pagamentos["formato_valor"].value_counts().to_dict())
print(f"valor que não converte: {nao_converte:,} | valor fora dos preços conhecidos: {fora_preco.sum():,}")
print(pd.crosstab(pagamentos["valor_num"], pagamentos["status"]))

# Em que período e em que cobranças aparece o formato R$ 0,00?
eh_loja = pagamentos["metodo"].isin({"app_store", "google_play"})
ultimo_dia_rs = pagamentos.loc[eh_br, "data_hora"].str[:10].max()
site_no_periodo = ~eh_loja & (pagamentos["data_hora"].str[:10] <= ultimo_dia_rs)
excecoes = pagamentos.loc[site_no_periodo & ~eh_br]
print(f"formato R$ 0,00: só no site ({sorted(pagamentos.loc[eh_br, 'metodo'].unique())}), "
      f"de {pagamentos.loc[eh_br, 'data_hora'].min()[:10]} a {ultimo_dia_rs}; nenhuma depois")
print(f"cobranças do site até {ultimo_dia_rs} no formato 0.00: {len(excecoes):,}, status: {excecoes['status'].value_counts().to_dict()}")
achado("Validade", "valor em dois formatos (0.00 e R$ 0,00)",
       f"{eh_br.sum():,} linhas em R$ 0,00: todas as cobranças do site até {ultimo_dia_rs} "
       f"(as {len(excecoes):,} do período em 0.00 são {', '.join(excecoes['status'].unique())}); "
       f"nenhuma depois, o que sugere mudança no sistema do site em julho de 2025. Todas convertem ({nao_converte:,} falhas)",
       eh_br.sum(),
       "Converter os dois formatos para número antes de somar; somar o texto daria erro ou ignoraria essas linhas")
achado("Validade", "valor fora dos preços conhecidos (34,90, 39,90, 299,90, 239,90)", f"{fora_preco.sum():,} linhas",
       fora_preco.sum(), "Nenhum")

formatos de valor: {'0.00': 145798, 'R$ 0,00': 19750}
valor que não converte: 0 | valor fora dos preços conhecidos: 0
status     aprovado  chargeback  estornado  recusado
valor_num                                           
-299.9            0          90         72         0
-39.9             0         206          0         0
 34.9        101278           0          0      1976
 39.9         43641           0          0       991
 239.9         1686           0          0        24
 299.9        15325           0          0       259


formato R$ 0,00: só no site (['cartao', 'pix_automatico']), de 2024-07-02 a 2025-06-30; nenhuma depois
cobranças do site até 2025-06-30 no formato 0.00: 44, status: {'chargeback': 41, 'estornado': 3}


In [11]:
padrao = pagamentos["data_hora"].str.replace(r"\d", "9", regex=True)
print("padrões de data_hora:", padrao.value_counts().to_dict())
data_invalida = int(pagamentos["dh_br"].isna().sum())
achado("Validade", "data_hora que não converte em data", f"{data_invalida:,} linhas", data_invalida, "Nenhum")

id_fora = ~pagamentos["pagamento_id"].str.fullmatch(r"p_\d{7}")
user_fora = ~pagamentos["user_id"].str.fullmatch(r"u_[0-9a-f]{8}")
sem_cadastro = ~pagamentos["user_id"].isin(usuarios["user_id"])
print(f"pagamento_id fora do padrão p_0000000: {id_fora.sum():,} | user_id fora do padrão: {user_fora.sum():,} | "
      f"sem cadastro: {sem_cadastro.sum():,}")
achado("Validade", "identificador fora do padrão ou user_id sem cadastro",
       f"{id_fora.sum():,} pagamento_id; {user_fora.sum():,} user_id fora do padrão; {sem_cadastro.sum():,} sem cadastro",
       (id_fora | user_fora | sem_cadastro).sum(), "Nenhum")

padrões de data_hora: {'9999-99-99T99:99:99Z': 112126, '9999-99-99 99:99:99': 53422}


pagamento_id fora do padrão p_0000000: 0 | user_id fora do padrão: 0 | sem cadastro: 0


## 3. Consistência

### 3.1 Formatos × método

In [12]:
display(pd.crosstab(pagamentos["metodo"], pagamentos["formato_data"]))
formato_trocado = ((eh_loja & (pagamentos["formato_data"] != "UTC (Z)")) |
                   (~eh_loja & (pagamentos["formato_data"] != "Brasília"))).sum()
achado("Consistência", "formato de data diferente do esperado para o método", f"{formato_trocado:,} linhas",
       formato_trocado, "Nenhum")

display(pd.crosstab(pagamentos["metodo"], pagamentos["formato_valor"]))

formato_data,Brasília,UTC (Z)
metodo,,
app_store,0,57020
cartao,34676,0
google_play,0,55106
pix_automatico,18746,0


formato_valor,0.00,"R$ 0,00"
metodo,,
app_store,57020,0
cartao,21632,13044
google_play,55106,0
pix_automatico,12040,6706


### 3.2 Sinal do valor × status

Chargeback e estorno devolvem dinheiro (valor negativo); aprovado e recusado são cobranças (positivo).

In [13]:
sinal = pagamentos["valor_num"].gt(0).map({True: "positivo", False: "negativo ou zero"})
display(pd.crosstab(pagamentos["status"], sinal))
DEVOLVE = {"chargeback", "estornado"}
sinal_trocado = ((pagamentos["status"].isin(DEVOLVE) & (pagamentos["valor_num"] >= 0)) |
                 (~pagamentos["status"].isin(DEVOLVE) & (pagamentos["valor_num"] <= 0))).sum()
achado("Consistência", "sinal do valor incoerente com o status", f"{sinal_trocado:,} linhas", sinal_trocado, "Nenhum")

valor_num,negativo ou zero,positivo
status,,
aprovado,0,161930
chargeback,296,0
estornado,72,0
recusado,0,3250


### 3.3 Valor devolvido × valor cobrado

Um chargeback ou estorno deveria devolver o valor da cobrança que está sendo contestada.
Comparamos com a última cobrança aprovada do mesmo usuário antes da devolução.

In [14]:
ultima_aprovada = pagamentos["valor_num"].where(pagamentos["status"] == "aprovado").groupby(pagamentos["user_id"]).ffill()
pagamentos["valor_cobrado_antes"] = ultima_aprovada.groupby(pagamentos["user_id"]).shift()
dev = pagamentos[pagamentos["status"].isin(DEVOLVE)].copy()
dev["bate"] = (-dev["valor_num"]).round(2) == dev["valor_cobrado_antes"].round(2)
display(pd.crosstab([dev["valor_num"]], dev["valor_cobrado_antes"].rename("última cobrança aprovada")))
divergente = dev[~dev["bate"]]
print(f"devoluções com valor diferente da cobrança: {len(divergente):,} de {len(dev):,}")
print(f"diferença total (devolvido a mais): R$ {(-divergente['valor_num'] - divergente['valor_cobrado_antes']).sum():,.2f}")
achado("Consistência", "chargeback ou estorno com valor diferente do que foi cobrado",
       f"{len(divergente):,} de {len(dev):,}: devolvem o preço de tabela (-39,90 sobre cobrança de 34,90; "
       "-299,90 sobre cobrança de 239,90 com cupom)",
       len(divergente),
       "Na conta de receita, usar como devolução o valor da cobrança original, não o valor da linha negativa")

última cobrança aprovada,34.9,39.9,239.9,299.9
valor_num,,,,
-299.9,0,0,41,121
-39.9,138,68,0,0


devoluções com valor diferente da cobrança: 179 de 368
diferença total (devolvido a mais): R$ 3,150.00


### 3.4 Preço do mensal × data do reajuste

R$ 34,90 até 28/02/2026; R$ 39,90 a partir de 01/03/2026.

In [15]:
mensal_positivo = pagamentos[pagamentos["valor_num"].isin([34.90, 39.90])]
display(pd.crosstab(mensal_positivo["valor_num"], (mensal_positivo["dh_br"] >= REAJUSTE).map({False: "antes", True: "depois"})))
antigo_depois = mensal_positivo[(mensal_positivo["valor_num"] == 34.90) & (mensal_positivo["dh_br"] >= REAJUSTE)]
novo_antes = mensal_positivo[(mensal_positivo["valor_num"] == 39.90) & (mensal_positivo["dh_br"] < REAJUSTE)]
print(f"R$ 34,90 depois do reajuste: {len(antigo_depois):,}, entre {antigo_depois['dh_br'].min():%d/%m} e "
      f"{antigo_depois['dh_br'].max():%d/%m/%Y}; por método: {antigo_depois['metodo'].value_counts().to_dict()}")
print(f"R$ 39,90 antes do reajuste: {len(novo_antes):,}")
achado("Consistência", "mensal cobrado pelo preço antigo depois do reajuste",
       f"{len(antigo_depois):,} cobranças de R$ 34,90 entre 01/03 e 12/03/2026; {len(novo_antes):,} de R$ 39,90 antes",
       len(antigo_depois) + len(novo_antes),
       "Poucos casos, nos primeiros dias de março: provável transição de preço. Manter e citar ao analisar o reajuste",
       status="esperado")

dh_br,antes,depois
valor_num,,
34.9,103238,16
39.9,0,44632


R$ 34,90 depois do reajuste: 16, entre 01/03 e 12/03/2026; por método: {'cartao': 6, 'google_play': 5, 'app_store': 4, 'pix_automatico': 1}
R$ 39,90 antes do reajuste: 0


### 3.5 Valor e método × plano e canal do evento no mesmo dia

In [16]:
METODO_CANAL = {"app_store": "app_store", "google_play": "google_play", "cartao": "web_cartao", "pix_automatico": "web_pix"}
m = aprovados[["user_id", "data_br", "valor_num", "metodo"]].reset_index().merge(
    ev_cobranca[["user_id", "data_br", "plano", "canal_cobranca"]].rename(columns={"data_br": "data_ev"}), on="user_id")
m = m[(m["data_ev"] - m["data_br"]).dt.days.abs() <= 1].drop_duplicates("index")
display(pd.crosstab(m["valor_num"], m["plano"]))
PLANO_DO_VALOR = {34.90: "mensal", 39.90: "mensal", 299.90: "anual", 239.90: "anual"}
plano_diverge = (m["valor_num"].map(PLANO_DO_VALOR) != m["plano"]).sum()
canal_diverge = (m["metodo"].map(METODO_CANAL) != m["canal_cobranca"]).sum()
print(f"valor incoerente com o plano: {plano_diverge:,} | método diferente do canal: {canal_diverge:,} (de {len(m):,})")
achado("Consistência", "valor incoerente com o plano do evento", f"{plano_diverge:,} de {len(m):,}", plano_diverge, "Nenhum")
achado("Consistência", "método diferente do canal do evento", f"{canal_diverge:,} de {len(m):,}", canal_diverge, "Nenhum")

plano,anual,mensal
valor_num,,
34.9,0,101278
39.9,0,43641
239.9,1686,0
299.9,15325,0


valor incoerente com o plano: 0 | método diferente do canal: 0 (de 161,930)


### 3.6 Recusa, nova tentativa e a coluna `tentativa`

Pelo dicionário, quando uma cobrança é recusada o sistema tenta de novo nos dias seguintes, e cada tentativa é uma linha com
o número da tentativa. Então se espera: recusada com `tentativa = 1`, seguida de uma nova linha com `tentativa = 2`.

In [17]:
g = pagamentos.groupby("user_id")
pagamentos["status_anterior"] = g["status"].shift()
pagamentos["status_seguinte"] = g["status"].shift(-1)
pagamentos["tentativa_seguinte"] = g["tentativa"].shift(-1)
pagamentos["dias_ate_seguinte"] = (g["dh_br"].shift(-1) - pagamentos["dh_br"]).dt.total_seconds() / 86400

display(pd.crosstab(pagamentos["status"], pagamentos["tentativa"]))

recusas = pagamentos[pagamentos["status"] == "recusado"]
nova_tentativa = recusas[(recusas["status_seguinte"] == "aprovado") & (recusas["dias_ate_seguinte"] <= 16)]
print(f"recusas: {len(recusas):,}; seguidas de cobrança aprovada em até 16 dias: {len(nova_tentativa):,}")
print("  tentativa registrada nessa nova cobrança:", nova_tentativa["tentativa_seguinte"].value_counts().to_dict())
print("  dias entre a recusa e a nova cobrança:", nova_tentativa["dias_ate_seguinte"].describe().round(1).to_dict())

t2 = pagamentos[pagamentos["tentativa"] == "2"]
print(f"linhas com tentativa = 2: {len(t2):,}; precedidas de recusa: {(t2['status_anterior'] == 'recusado').sum():,}")

tentativa,1,2
status,,
aprovado,157064,4866
chargeback,296,0
estornado,72,0
recusado,3250,0


recusas: 3,250; seguidas de cobrança aprovada em até 16 dias: 1,881
  tentativa registrada nessa nova cobrança: {'1': 1827, '2': 54}
  dias entre a recusa e a nova cobrança: {'count': 1881.0, 'mean': 5.2, 'std': 3.7, 'min': 0.3, '25%': 2.5, '50%': 4.4, '75%': 6.2, 'max': 15.5}
linhas com tentativa = 2: 4,866; precedidas de recusa: 54


In [18]:
nova_como_1 = int((nova_tentativa["tentativa_seguinte"] == "1").sum())
t2_sem_recusa = int((t2["status_anterior"] != "recusado").sum())
achado("Consistência", "coluna tentativa não marca a nova tentativa depois de uma recusa",
       f"{nova_como_1:,} de {len(nova_tentativa):,} novas cobranças após recusa vêm com tentativa = 1; "
       f"{t2_sem_recusa:,} de {len(t2):,} linhas com tentativa = 2 não têm recusa antes",
       nova_como_1 + t2_sem_recusa,
       "Não usar a coluna tentativa. Identificar a nova tentativa pela sequência: recusado seguido de aprovado do mesmo "
       "usuário em até 16 dias. O significado de tentativa = 2 não está no dicionário: perguntar ao dono do dado")

### 3.7 Desfecho de cada recusa e período de carência

In [19]:
expiracoes = eventos[eventos["evento"] == "assinatura_expirada"][["user_id", "dh_br"]]
m = recusas[["user_id", "dh_br"]].reset_index().merge(expiracoes.rename(columns={"dh_br": "dh_exp"}), on="user_id")
m["dias"] = (m["dh_exp"] - m["dh_br"]).dt.total_seconds() / 86400
expira_depois = m[(m["dias"] >= 0) & (m["dias"] <= 40)].groupby("index")["dias"].min()

desfecho = pd.Series("sem desfecho", index=recusas.index)
desfecho[expira_depois.index] = "assinatura expirou (até 40 dias)"
desfecho[nova_tentativa.index] = "nova cobrança aprovada (até 16 dias)"
recente = recusas["dh_br"] >= FIM_CORTE - pd.Timedelta(days=16)
desfecho[(desfecho == "sem desfecho") & recente] = "em andamento (recusa nos últimos 16 dias)"
display(desfecho.value_counts().to_frame("recusas"))

carencia = eventos[eventos["evento"] == "periodo_carencia"]
recusa_com_carencia = casa_por_data(recusas, carencia)
carencia_com_recusa = casa_por_data(carencia, recusas)
print(f"carências com recusa no mesmo dia: {carencia_com_recusa.sum():,} de {len(carencia):,}")
print(f"recusas com carência no mesmo dia: {recusa_com_carencia.sum():,} de {len(recusas):,}")
print("  % com carência por método:", recusa_com_carencia.groupby(recusas["metodo"]).mean().round(2).to_dict())

sem_desfecho = int((desfecho == "sem desfecho").sum())
achado("Consistência", "recusa sem desfecho (nem nova cobrança nem expiração)", f"{sem_desfecho:,} de {len(recusas):,}",
       sem_desfecho, "Se houver: olhar a sequência do usuário")
achado("Consistência", "período de carência sem recusa no mesmo dia",
       f"{(~carencia_com_recusa).sum():,} de {len(carencia):,}", (~carencia_com_recusa).sum(), "Nenhum")

,recusas
nova cobrança aprovada (até 16 dias),1881
assinatura expirou (até 40 dias),1321
em andamento (recusa nos últimos 16 dias),48


carências com recusa no mesmo dia: 1,346 de 1,346
recusas com carência no mesmo dia: 1,346 de 3,250
  % com carência por método: {'app_store': 0.45, 'cartao': 0.4, 'google_play': 0.41, 'pix_automatico': 0.42}


## 4. Unicidade

O `pagamento_id` é único por construção. A pergunta que importa é se a mesma cobrança aparece em duas linhas
com identificadores diferentes.

In [20]:
dup_id = int(pagamentos["pagamento_id"].duplicated().sum())
dup_linha = int(pagamentos.duplicated(subset=colunas).sum())
CHAVE = ["user_id", "data_hora", "valor_num", "metodo", "status", "tentativa"]
dup_sem_id = pagamentos.duplicated(subset=CHAVE, keep="first")
print(f"pagamento_id repetido: {dup_id:,} | linha inteira repetida: {dup_linha:,}")
print(f"mesma cobrança com pagamento_id diferente (usuário, instante, valor, método, status, tentativa): {dup_sem_id.sum():,}")

copias = pagamentos[dup_sem_id]
originais = pagamentos[pagamentos.duplicated(subset=CHAVE, keep="last")]
distancia_id = (copias["pagamento_id"].str[2:].astype(int).values - originais["pagamento_id"].str[2:].astype(int).values)
print("  status:", copias["status"].value_counts().to_dict(), "| método:", copias["metodo"].value_counts().to_dict())
print("  distância entre os pagamento_id da cópia e do original:", pd.Series(distancia_id).value_counts().to_dict())
print(f"  valor somado das cópias: R$ {copias['valor_num'].sum():,.2f}")
display(pd.concat([originais.head(3), copias.head(3)]).sort_values(["user_id", "pagamento_id"])[colunas])

achado("Unicidade", "pagamento_id repetido ou linha inteira repetida", f"{dup_id:,} e {dup_linha:,}", dup_id + dup_linha,
       "Nenhum")
achado("Unicidade", "mesma cobrança gravada duas vezes com pagamento_id seguido",
       f"{dup_sem_id.sum():,} cópias ({dup_sem_id.sum() / len(aprovados):.1%} dos aprovados), "
       f"R$ {copias['valor_num'].sum():,.2f} a mais; explica a diferença entre pagamentos aprovados e inícios + renovações",
       dup_sem_id.sum(),
       "Remover a cópia (manter o menor pagamento_id) antes de somar receita ou contar cobranças")

pagamento_id repetido: 0 | linha inteira repetida: 0
mesma cobrança com pagamento_id diferente (usuário, instante, valor, método, status, tentativa): 1,941
  status: {'aprovado': 1941} | método: {'app_store': 663, 'google_play': 639, 'cartao': 384, 'pix_automatico': 255}
  distância entre os pagamento_id da cópia e do original: {1: 1941}
  valor somado das cópias: R$ 118,295.90


,pagamento_id,user_id,data_hora,valor,metodo,status,tentativa
405,p_0000405,u_0074af35,2024-07-06T17:06:04Z,34.90,google_play,aprovado,2
406,p_0000406,u_0074af35,2024-07-06T17:06:04Z,34.90,google_play,aprovado,2
122201,p_0122201,u_00837de5,2026-03-26 20:27:40,39.90,cartao,aprovado,2
122202,p_0122202,u_00837de5,2026-03-26 20:27:40,39.90,cartao,aprovado,2
70988,p_0070988,u_00a06f1a,2025-08-20 21:04:08,34.90,cartao,aprovado,1
70989,p_0070989,u_00a06f1a,2025-08-20 21:04:08,34.90,cartao,aprovado,1


In [21]:
por_usuario = (aprovados.groupby("user_id").size()
               - ev_cobranca.groupby("user_id").size().reindex(aprovados["user_id"].unique()).fillna(0))
print(f"aprovados - (inícios + renovações), somado por usuário: {int(por_usuario.sum()):,} "
      f"| cópias encontradas: {int(dup_sem_id.sum()):,}")

aprovados - (inícios + renovações), somado por usuário: 1,941 | cópias encontradas: 1,941


## 5. Temporalidade

### 5.1 Período coberto e efeito do fuso

In [22]:
print(f"primeira cobrança: {pagamentos['dh_br'].min()} | última: {pagamentos['dh_br'].max()} (horário de Brasília)")
antes = int((pagamentos["dh_br"] < INICIO_LOG).sum())
depois = int((pagamentos["dh_br"] >= FIM_CORTE).sum())
data_texto = pd.to_datetime(pagamentos["data_hora"].str[:10], format="%Y-%m-%d")
depois_texto = int((data_texto > CORTE).sum())
vira_dia = int((pagamentos["data_br"] != data_texto).sum())
print(f"antes de 01/07/2024: {antes:,} | depois do corte: {depois:,} | 'depois do corte' lendo só o texto: {depois_texto:,}")
achado("Temporalidade", "cobranças fora do período 01/07/2024 a 30/09/2026", f"{antes:,} antes e {depois:,} depois",
       antes + depois, "Nenhum")
achado("Temporalidade", "cobranças que mudam de dia ao acertar o fuso",
       f"{vira_dia:,} linhas; lendo só o texto, {depois_texto:,} pareceriam posteriores ao corte", vira_dia,
       "Usar a data convertida para Brasília (decisão já registrada para eventos_assinatura.csv)", status="esperado")

primeira cobrança: 2024-07-02 06:14:11 | última: 2026-09-30 22:54:51 (horário de Brasília)
antes de 01/07/2024: 0 | depois do corte: 0 | 'depois do corte' lendo só o texto: 30


### 5.2 Formato do valor ao longo do tempo

O achado está registrado em Validade; a tabela mostra a virada em julho de 2025.

In [23]:
pd.crosstab(pagamentos.loc[~eh_loja, "mes_br"], pagamentos.loc[~eh_loja, "formato_valor"])

formato_valor,0.00,"R$ 0,00"
mes_br,,
2024-07,0,1103
2024-08,2,1278
2024-09,8,1365
2024-10,1,1506
2024-11,5,1559
2024-12,2,1604
2025-01,7,1820
2025-02,5,1694
2025-03,3,1977


### 5.3 Volume e receita por mês

In [24]:
sem_copias = pagamentos[~dup_sem_id]
mensal = sem_copias.pivot_table(index="mes_br", columns="status", values="valor_num", aggfunc="size", fill_value=0)
mensal["receita_aprovada"] = sem_copias[sem_copias["status"] == "aprovado"].groupby("mes_br")["valor_num"].sum()
mensal["devolvido"] = -sem_copias[sem_copias["status"].isin(DEVOLVE)].groupby("mes_br")["valor_num"].sum()
mensal["% recusado"] = (100 * mensal["recusado"] / (mensal["aprovado"] + mensal["recusado"])).round(1)
display(mensal.fillna(0).round(2))

meses_esperados = pd.period_range("2024-07", "2026-09", freq="M")
meses_faltando = meses_esperados.difference(mensal.index)
saltos = mensal["aprovado"].pct_change()
saltos = saltos[saltos.abs() > 0.5]
print(f"meses sem cobrança: {list(meses_faltando.astype(str))} | variação acima de 50%: {len(saltos)}")
achado("Temporalidade", "meses sem cobrança ou com salto acima de 50%",
       f"{len(meses_faltando)} meses vazios; {len(saltos)} saltos", len(meses_faltando) + len(saltos), "Nenhum")

status,aprovado,chargeback,estornado,recusado,receita_aprovada,devolvido,% recusado
mes_br,,,,,,,
2024-07,3314,1,0,56,221788.6,39.9,1.7
2024-08,3835,7,0,58,265181.5,539.3,1.5
2024-09,4041,16,0,62,285345.9,2198.4,1.5
2024-10,4440,6,0,85,294911.0,499.4,1.9
2024-11,4580,10,1,71,297687.0,1738.9,1.5
2024-12,4786,7,0,76,290276.4,1059.3,1.6
2025-01,5439,16,1,90,366901.1,2498.3,1.6
2025-02,5113,17,1,99,329118.7,2538.2,1.9
2025-03,5803,9,0,100,359644.7,619.1,1.7


meses sem cobrança: [] | variação acima de 50%: 0


### 5.4 Ritmo das cobranças: a pergunta que ficou aberta nos eventos

Em `eventos_assinatura.csv`, 2.225 renovações caíram fora do intervalo esperado, com um ciclo longo seguido de um curto.
A hipótese era cobrança recusada e aprovada na nova tentativa. Aqui dá para conferir.

In [25]:
aprov = sem_copias[sem_copias["status"] == "aprovado"].copy()
recusas_acum = (sem_copias["status"] == "recusado").groupby(sem_copias["user_id"]).cumsum()
aprov["dias"] = aprov.groupby("user_id")["dh_br"].diff().dt.total_seconds() / 86400
aprov["recusa_entre"] = recusas_acum[aprov.index].groupby(aprov["user_id"]).diff().gt(0)
mensais = aprov[aprov["valor_num"].isin([34.90, 39.90])]
longos = mensais[mensais["dias"] > 33]
normais = mensais[mensais["dias"].between(27, 33)]
print(f"ciclos mensais longos (> 33 dias): {len(longos):,}; com recusa entre as duas cobranças: {longos['recusa_entre'].sum():,}")
print(f"ciclos mensais normais (27 a 33 dias): {len(normais):,}; com recusa entre: {normais['recusa_entre'].sum():,}")
print("ciclos longos por método:", longos["metodo"].value_counts().to_dict())
achado("Temporalidade", "ciclo mensal longo explicado por recusa",
       f"{longos['recusa_entre'].sum():,} de {len(longos):,} ciclos longos têm recusa entre as cobranças: confirma a "
       "hipótese aberta em eventos_assinatura.csv",
       len(longos),
       "Não é erro: é a nova tentativa depois da recusa. Ao medir atraso de pagamento, usar a sequência recusado -> aprovado",
       status="esperado")

ciclos mensais longos (> 33 dias): 1,158; com recusa entre as duas cobranças: 1,158
ciclos mensais normais (27 a 33 dias): 121,544; com recusa entre: 532
ciclos longos por método: {'cartao': 508, 'google_play': 300, 'app_store': 237, 'pix_automatico': 113}


## 6. Resumo

In [26]:
resumo = pd.DataFrame(achados)
resumo["% do arquivo"] = (100 * resumo["linhas_afetadas"] / N).round(2)
resumo.to_csv(SAIDAS / "qualidade_pagamentos.csv", index=False)
print(resumo["status"].value_counts().to_dict())
resumo

{'ok': 20, 'atenção': 5, 'esperado': 3}


,criterio,checagem,status,resultado,linhas_afetadas,tratamento_sugerido,% do arquivo
0,Completude,campos vazios,ok,0 células vazias,0,Nenhum,0.00
1,Completude,início ou renovação sem pagamento aprovado,ok,"0 de 159,989 eventos",0,Nenhum,0.00
2,Completude,pagamento aprovado sem início ou renovação,ok,"0 de 161,930 pagamentos",0,Nenhum,0.00
3,Completude,chargeback ou estorno sem evento correspondente,ok,chargeback: 296 de 296; estornado: 72 de 72,0,Nenhum,0.00
4,Completude,cobrança com o desconto de 30% da campanha de ...,atenção,0 cobranças; nenhum valor diferente dos preços...,0,Não dá para medir o custo do cupom da campanha...,0.00
5,Completude,pagamentos de contas de teste,ok,0,0,Se houver: excluir,0.00
6,Validade,metodo fora da lista do dicionário,ok,0 linhas,0,Nenhum,0.00
7,Validade,status fora da lista do dicionário,ok,0 linhas,0,Nenhum,0.00
8,Validade,tentativa que não é inteiro positivo,ok,0 linhas,0,Nenhum,0.00
9,Validade,"valor em dois formatos (0.00 e R$ 0,00)",atenção,"19,750 linhas em R$ 0,00: todas as cobranças d...",19750,Converter os dois formatos para número antes d...,11.93


In [27]:
hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados/brutos/ inalterado: True
